In [1]:
import re
import os

In [2]:
print(os.getcwd())

d:\MSc Artificial Intelligence\COMP5012


In [3]:
file_path = "data/Modules (data for JAN assessment).txt"

In [4]:
def normalise_module_code(code: str) -> str:
    code = code.strip().upper()
    code = code.replace("M0D", "MOD")

    import re
    match = re.match(r"MOD(\d+)$", code)
    if match:
        return f"MOD{match.group(1).zfill(3)}"

    digits = re.findall(r"\d+", code)
    if digits:
        return f"MOD{digits[0].zfill(3)}"

    return code


def parse_module_line(line: str) -> dict:
    parts = line.strip().split("|")

    module_code = normalise_module_code(parts[0])
    staff_name = parts[1].strip()
    num_labs = int(parts[2].strip())

    conflicts = [
        normalise_module_code(c)
        for c in parts[3].split(",")
        if c.strip()
    ]

    return {
        "module_id": module_code,
        "staff": staff_name,
        "num_labs": num_labs,
        "conflicts": conflicts
    }


def load_modules(file_path):
    modules = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                modules.append(parse_module_line(line))

    return modules

In [5]:
modules = load_modules(file_path)

print(f"Loaded {len(modules)} modules")
modules[:3]

Loaded 17 modules


[{'module_id': 'MOD001',
  'staff': 'Zacharias Karstensen',
  'num_labs': 2,
  'conflicts': ['MOD002',
   'MOD003',
   'MOD004',
   'MOD005',
   'MOD006',
   'MOD007',
   'MOD008',
   'MOD009',
   'MOD010',
   'MOD013']},
 {'module_id': 'MOD002',
  'staff': 'Dominykas Cleary',
  'num_labs': 2,
  'conflicts': ['MOD001',
   'MOD003',
   'MOD004',
   'MOD005',
   'MOD006',
   'MOD007',
   'MOD008',
   'MOD009',
   'MOD010',
   'MOD013']},
 {'module_id': 'MOD003',
  'staff': 'Zacharias Karstensen',
  'num_labs': 2,
  'conflicts': ['MOD001',
   'MOD002',
   'MOD004',
   'MOD005',
   'MOD006',
   'MOD007',
   'MOD008',
   'MOD009',
   'MOD010',
   'MOD011',
   'MOD012',
   'MOD013']}]

In [6]:
def build_events(modules):
    """
    Convert module data into a flat list of teaching events.
    
    Each module creates:
    - 1 lecture event
    - num_labs lab events
    """
    events = []

    for module in modules:
        module_id = module["module_id"]
        staff = module["staff"]

        # Add lecture event
        events.append({
            "event_id": f"{module_id}_LEC",
            "module_id": module_id,
            "staff": staff,
            "event_type": "lecture"
        })

        # Add lab events
        for lab_num in range(1, module["num_labs"] + 1):
            events.append({
                "event_id": f"{module_id}_LAB{lab_num}",
                "module_id": module_id,
                "staff": staff,
                "event_type": "lab"
            })

    return events

In [7]:
events = build_events(modules)

print(f"Total number of events: {len(events)}")
events[:10]

Total number of events: 48


[{'event_id': 'MOD001_LEC',
  'module_id': 'MOD001',
  'staff': 'Zacharias Karstensen',
  'event_type': 'lecture'},
 {'event_id': 'MOD001_LAB1',
  'module_id': 'MOD001',
  'staff': 'Zacharias Karstensen',
  'event_type': 'lab'},
 {'event_id': 'MOD001_LAB2',
  'module_id': 'MOD001',
  'staff': 'Zacharias Karstensen',
  'event_type': 'lab'},
 {'event_id': 'MOD002_LEC',
  'module_id': 'MOD002',
  'staff': 'Dominykas Cleary',
  'event_type': 'lecture'},
 {'event_id': 'MOD002_LAB1',
  'module_id': 'MOD002',
  'staff': 'Dominykas Cleary',
  'event_type': 'lab'},
 {'event_id': 'MOD002_LAB2',
  'module_id': 'MOD002',
  'staff': 'Dominykas Cleary',
  'event_type': 'lab'},
 {'event_id': 'MOD003_LEC',
  'module_id': 'MOD003',
  'staff': 'Zacharias Karstensen',
  'event_type': 'lecture'},
 {'event_id': 'MOD003_LAB1',
  'module_id': 'MOD003',
  'staff': 'Zacharias Karstensen',
  'event_type': 'lab'},
 {'event_id': 'MOD003_LAB2',
  'module_id': 'MOD003',
  'staff': 'Zacharias Karstensen',
  'event_t

In [8]:
for event in events:
    print(event)

{'event_id': 'MOD001_LEC', 'module_id': 'MOD001', 'staff': 'Zacharias Karstensen', 'event_type': 'lecture'}
{'event_id': 'MOD001_LAB1', 'module_id': 'MOD001', 'staff': 'Zacharias Karstensen', 'event_type': 'lab'}
{'event_id': 'MOD001_LAB2', 'module_id': 'MOD001', 'staff': 'Zacharias Karstensen', 'event_type': 'lab'}
{'event_id': 'MOD002_LEC', 'module_id': 'MOD002', 'staff': 'Dominykas Cleary', 'event_type': 'lecture'}
{'event_id': 'MOD002_LAB1', 'module_id': 'MOD002', 'staff': 'Dominykas Cleary', 'event_type': 'lab'}
{'event_id': 'MOD002_LAB2', 'module_id': 'MOD002', 'staff': 'Dominykas Cleary', 'event_type': 'lab'}
{'event_id': 'MOD003_LEC', 'module_id': 'MOD003', 'staff': 'Zacharias Karstensen', 'event_type': 'lecture'}
{'event_id': 'MOD003_LAB1', 'module_id': 'MOD003', 'staff': 'Zacharias Karstensen', 'event_type': 'lab'}
{'event_id': 'MOD003_LAB2', 'module_id': 'MOD003', 'staff': 'Zacharias Karstensen', 'event_type': 'lab'}
{'event_id': 'MOD004_LEC', 'module_id': 'MOD004', 'staff':

In [9]:
num_lectures = sum(1 for event in events if event["event_type"] == "lecture")
num_labs = sum(1 for event in events if event["event_type"] == "lab")

print("Number of lectures:", num_lectures)
print("Number of labs:", num_labs)
print("Total events:", len(events))

Number of lectures: 17
Number of labs: 31
Total events: 48


In [10]:
DAYS_PER_WEEK = 5
SLOTS_PER_DAY = 4
TOTAL_SLOTS = DAYS_PER_WEEK * SLOTS_PER_DAY

DAY_NAMES = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
TIME_LABELS = ["09:00-11:00", "11:00-13:00", "14:00-16:00", "16:00-18:00"]

print("Total timetable slots:", TOTAL_SLOTS)

Total timetable slots: 20


In [11]:
def slot_to_day(slot):
    """
    Convert a slot number into a day index.
    Example:
    0-3   -> Monday
    4-7   -> Tuesday
    8-11  -> Wednesday
    12-15 -> Thursday
    16-19 -> Friday
    """
    return slot // SLOTS_PER_DAY


def slot_to_time_index(slot):
    """
    Convert a slot number into a time index within the day.
    """
    return slot % SLOTS_PER_DAY


def slot_to_label(slot):
    """
    Convert a slot number into a readable label like:
    'Monday 09:00-11:00'
    """
    day_name = DAY_NAMES[slot_to_day(slot)]
    time_label = TIME_LABELS[slot_to_time_index(slot)]
    return f"{day_name} {time_label}"

In [12]:
for slot in range(TOTAL_SLOTS):
    print(slot, "->", slot_to_label(slot))

0 -> Monday 09:00-11:00
1 -> Monday 11:00-13:00
2 -> Monday 14:00-16:00
3 -> Monday 16:00-18:00
4 -> Tuesday 09:00-11:00
5 -> Tuesday 11:00-13:00
6 -> Tuesday 14:00-16:00
7 -> Tuesday 16:00-18:00
8 -> Wednesday 09:00-11:00
9 -> Wednesday 11:00-13:00
10 -> Wednesday 14:00-16:00
11 -> Wednesday 16:00-18:00
12 -> Thursday 09:00-11:00
13 -> Thursday 11:00-13:00
14 -> Thursday 14:00-16:00
15 -> Thursday 16:00-18:00
16 -> Friday 09:00-11:00
17 -> Friday 11:00-13:00
18 -> Friday 14:00-16:00
19 -> Friday 16:00-18:00


In [13]:
import random

In [14]:
def create_random_chromosome(events, total_slots=TOTAL_SLOTS):
    """
    Create one random timetable chromosome.
    Each gene is a slot number assigned to one event.
    """
    chromosome = [random.randint(0, total_slots - 1) for _ in events]
    return chromosome

In [15]:
chromosome = create_random_chromosome(events)

print("Chromosome length:", len(chromosome))
print("First 10 genes:", chromosome[:10])

Chromosome length: 48
First 10 genes: [10, 2, 7, 2, 13, 1, 1, 5, 0, 8]


In [16]:
def decode_chromosome(chromosome, events):
    """
    Convert a chromosome into a readable list of event assignments.
    """
    decoded = []

    for event, slot in zip(events, chromosome):
        decoded.append({
            "event_id": event["event_id"],
            "module_id": event["module_id"],
            "staff": event["staff"],
            "event_type": event["event_type"],
            "slot": slot,
            "slot_label": slot_to_label(slot)
        })

    return decoded

In [17]:
decoded = decode_chromosome(chromosome, events)

for row in decoded[:10]:
    print(row)

{'event_id': 'MOD001_LEC', 'module_id': 'MOD001', 'staff': 'Zacharias Karstensen', 'event_type': 'lecture', 'slot': 10, 'slot_label': 'Wednesday 14:00-16:00'}
{'event_id': 'MOD001_LAB1', 'module_id': 'MOD001', 'staff': 'Zacharias Karstensen', 'event_type': 'lab', 'slot': 2, 'slot_label': 'Monday 14:00-16:00'}
{'event_id': 'MOD001_LAB2', 'module_id': 'MOD001', 'staff': 'Zacharias Karstensen', 'event_type': 'lab', 'slot': 7, 'slot_label': 'Tuesday 16:00-18:00'}
{'event_id': 'MOD002_LEC', 'module_id': 'MOD002', 'staff': 'Dominykas Cleary', 'event_type': 'lecture', 'slot': 2, 'slot_label': 'Monday 14:00-16:00'}
{'event_id': 'MOD002_LAB1', 'module_id': 'MOD002', 'staff': 'Dominykas Cleary', 'event_type': 'lab', 'slot': 13, 'slot_label': 'Thursday 11:00-13:00'}
{'event_id': 'MOD002_LAB2', 'module_id': 'MOD002', 'staff': 'Dominykas Cleary', 'event_type': 'lab', 'slot': 1, 'slot_label': 'Monday 11:00-13:00'}
{'event_id': 'MOD003_LEC', 'module_id': 'MOD003', 'staff': 'Zacharias Karstensen', 'ev

In [18]:
import pandas as pd

decoded_df = pd.DataFrame(decoded)
decoded_df.head(10)

,event_id,module_id,staff,event_type,slot,slot_label
0,MOD001_LEC,MOD001,Zacharias Karstensen,lecture,10,Wednesday 14:00-16:00
1,MOD001_LAB1,MOD001,Zacharias Karstensen,lab,2,Monday 14:00-16:00
2,MOD001_LAB2,MOD001,Zacharias Karstensen,lab,7,Tuesday 16:00-18:00
3,MOD002_LEC,MOD002,Dominykas Cleary,lecture,2,Monday 14:00-16:00
4,MOD002_LAB1,MOD002,Dominykas Cleary,lab,13,Thursday 11:00-13:00
5,MOD002_LAB2,MOD002,Dominykas Cleary,lab,1,Monday 11:00-13:00
6,MOD003_LEC,MOD003,Zacharias Karstensen,lecture,1,Monday 11:00-13:00
7,MOD003_LAB1,MOD003,Zacharias Karstensen,lab,5,Tuesday 11:00-13:00
8,MOD003_LAB2,MOD003,Zacharias Karstensen,lab,0,Monday 09:00-11:00
9,MOD004_LEC,MOD004,Laila Deniau,lecture,8,Wednesday 09:00-11:00


In [19]:
module_conflicts = {
    module["module_id"]: set(module["conflicts"])
    for module in modules
}

module_conflicts

{'MOD001': {'MOD002',
  'MOD003',
  'MOD004',
  'MOD005',
  'MOD006',
  'MOD007',
  'MOD008',
  'MOD009',
  'MOD010',
  'MOD013'},
 'MOD002': {'MOD001',
  'MOD003',
  'MOD004',
  'MOD005',
  'MOD006',
  'MOD007',
  'MOD008',
  'MOD009',
  'MOD010',
  'MOD013'},
 'MOD003': {'MOD001',
  'MOD002',
  'MOD004',
  'MOD005',
  'MOD006',
  'MOD007',
  'MOD008',
  'MOD009',
  'MOD010',
  'MOD011',
  'MOD012',
  'MOD013'},
 'MOD004': {'MOD001',
  'MOD002',
  'MOD003',
  'MOD005',
  'MOD006',
  'MOD007',
  'MOD008',
  'MOD009',
  'MOD010',
  'MOD011',
  'MOD012',
  'MOD013'},
 'MOD005': {'MOD001',
  'MOD002',
  'MOD003',
  'MOD004',
  'MOD006',
  'MOD007',
  'MOD008',
  'MOD009',
  'MOD010',
  'MOD011',
  'MOD012'},
 'MOD006': {'MOD001',
  'MOD002',
  'MOD003',
  'MOD004',
  'MOD005',
  'MOD007',
  'MOD008',
  'MOD009',
  'MOD010',
  'MOD011',
  'MOD012'},
 'MOD007': {'MOD001',
  'MOD002',
  'MOD003',
  'MOD004',
  'MOD005',
  'MOD006',
  'MOD008',
  'MOD009',
  'MOD010',
  'MOD011',
  'MOD014',


In [20]:
def count_clashes(chromosome, events, module_conflicts):
    """
    Count clashes where two events from conflicting modules
    are scheduled in the same slot.
    
    A clash occurs if:
    - the two events belong to different modules
    - the modules are in each other's conflict list
    - both events are assigned the same slot
    """
    clashes = 0
    counted_pairs = set()

    for i in range(len(events)):
        module_i = events[i]["module_id"]
        slot_i = chromosome[i]

        for j in range(i + 1, len(events)):
            module_j = events[j]["module_id"]
            slot_j = chromosome[j]

            if module_i == module_j:
                continue

            # Check if these modules conflict
            if module_j in module_conflicts.get(module_i, set()):
                if slot_i == slot_j:
                    pair = tuple(sorted((module_i, module_j)) + [slot_i])

                    if pair not in counted_pairs:
                        clashes += 1
                        counted_pairs.add(pair)

    return clashes

In [21]:
def count_staff_teaching_days(chromosome, events):
    """
    Count the total number of unique teaching days across all staff.
    Lower is better because staff teaching should be compressed
    into fewer days.
    """
    staff_days = {}

    for event, slot in zip(events, chromosome):
        staff = event["staff"]
        day = slot_to_day(slot)

        if staff not in staff_days:
            staff_days[staff] = set()

        staff_days[staff].add(day)

    total_days = sum(len(days) for days in staff_days.values())
    return total_days

In [22]:
def count_penalties(chromosome, events):
    """
    Count timetable feasibility violations.

    Penalty types:
    1. Same staff assigned to multiple events in the same slot
    2. Same module assigned to multiple events in the same slot
    """
    penalties = 0

    for i in range(len(events)):
        for j in range(i + 1, len(events)):
            same_slot = chromosome[i] == chromosome[j]

            if not same_slot:
                continue

            # Same staff overlap
            if events[i]["staff"] == events[j]["staff"]:
                penalties += 1

            # Same module overlap
            if events[i]["module_id"] == events[j]["module_id"]:
                penalties += 1

    return penalties

In [23]:
def evaluate_chromosome(chromosome, events, module_conflicts, penalty_weight=100):
    """
    Evaluate one timetable chromosome.

    Returns:
    - objective_1: clashes + penalty
    - objective_2: staff teaching days + penalty
    - raw details for analysis
    """
    clashes = count_clashes(chromosome, events, module_conflicts)
    staff_days = count_staff_teaching_days(chromosome, events)
    penalties = count_penalties(chromosome, events)

    objective_1 = clashes + penalty_weight * penalties
    objective_2 = staff_days + penalty_weight * penalties

    return {
        "objective_1": objective_1,
        "objective_2": objective_2,
        "clashes": clashes,
        "staff_days": staff_days,
        "penalties": penalties
    }

In [24]:
fitness = evaluate_chromosome(chromosome, events, module_conflicts)

fitness

{'objective_1': 527,
 'objective_2': 527,
 'clashes': 27,
 'staff_days': 27,
 'penalties': 5}

In [25]:
print("Objective 1 (clashes + penalties):", fitness["objective_1"])
print("Objective 2 (staff days + penalties):", fitness["objective_2"])
print("Raw clashes:", fitness["clashes"])
print("Raw staff teaching days:", fitness["staff_days"])
print("Penalties:", fitness["penalties"])

Objective 1 (clashes + penalties): 527
Objective 2 (staff days + penalties): 527
Raw clashes: 27
Raw staff teaching days: 27
Penalties: 5


In [26]:
for i in range(5):
    test_chromosome = create_random_chromosome(events)
    test_fitness = evaluate_chromosome(test_chromosome, events, module_conflicts)
    print(f"Chromosome {i+1}: {test_fitness}")

Chromosome 1: {'objective_1': 1038, 'objective_2': 1029, 'clashes': 38, 'staff_days': 29, 'penalties': 10}
Chromosome 2: {'objective_1': 538, 'objective_2': 525, 'clashes': 38, 'staff_days': 25, 'penalties': 5}
Chromosome 3: {'objective_1': 1934, 'objective_2': 1925, 'clashes': 34, 'staff_days': 25, 'penalties': 19}
Chromosome 4: {'objective_1': 1337, 'objective_2': 1328, 'clashes': 37, 'staff_days': 28, 'penalties': 13}
Chromosome 5: {'objective_1': 839, 'objective_2': 827, 'clashes': 39, 'staff_days': 27, 'penalties': 8}


In [27]:
def create_individual(events, module_conflicts):
    """
    Create one random timetable individual and evaluate it.
    """
    chromosome = create_random_chromosome(events)
    fitness = evaluate_chromosome(chromosome, events, module_conflicts)

    individual = {
        "chromosome": chromosome,
        "objective_1": fitness["objective_1"],
        "objective_2": fitness["objective_2"],
        "clashes": fitness["clashes"],
        "staff_days": fitness["staff_days"],
        "penalties": fitness["penalties"]
    }

    return individual

In [28]:
def create_population(pop_size, events, module_conflicts):
    """
    Create a list of evaluated individuals.
    """
    population = [create_individual(events, module_conflicts) for _ in range(pop_size)]
    return population

In [29]:
POPULATION_SIZE = 50

population = create_population(POPULATION_SIZE, events, module_conflicts)

print("Population size:", len(population))
population[0]

Population size: 50


{'chromosome': [2,
  5,
  16,
  19,
  0,
  13,
  9,
  18,
  6,
  16,
  17,
  7,
  9,
  13,
  17,
  5,
  3,
  8,
  17,
  5,
  13,
  13,
  11,
  15,
  11,
  7,
  7,
  17,
  13,
  2,
  13,
  2,
  12,
  2,
  15,
  0,
  14,
  1,
  0,
  0,
  9,
  19,
  13,
  8,
  19,
  7,
  15,
  2],
 'objective_1': 929,
 'objective_2': 928,
 'clashes': 29,
 'staff_days': 28,
 'penalties': 9}

In [30]:
population_df = pd.DataFrame([
    {
        "objective_1": ind["objective_1"],
        "objective_2": ind["objective_2"],
        "clashes": ind["clashes"],
        "staff_days": ind["staff_days"],
        "penalties": ind["penalties"]
    }
    for ind in population
])

population_df.head(10)

,objective_1,objective_2,clashes,staff_days,penalties
0,929,928,29,28,9
1,1323,1330,23,30,13
2,726,728,26,28,7
3,1023,1028,23,28,10
4,1330,1326,30,26,13
5,943,924,43,24,9
6,532,530,32,30,5
7,1131,1128,31,28,11
8,934,926,34,26,9
9,635,626,35,26,6


In [31]:
population_df.describe()

,objective_1,objective_2,clashes,staff_days,penalties
count,50.000000,50.000000,50.000000,50.000000,50.000000
mean,1036.580000,1032.540000,30.580000,26.540000,10.060000
std,380.963981,382.168742,5.135272,1.554585,3.824785
min,329.000000,327.000000,20.000000,24.000000,3.000000
25%,756.000000,752.000000,27.000000,25.000000,7.250000
50%,1026.000000,1026.000000,31.000000,26.000000,10.000000
75%,1226.250000,1225.750000,34.000000,28.000000,12.000000
max,2120.000000,2127.000000,43.000000,30.000000,21.000000


In [32]:
best_obj1 = min(population, key=lambda ind: ind["objective_1"])
best_obj2 = min(population, key=lambda ind: ind["objective_2"])

print("Best by objective 1:")
print({
    "objective_1": best_obj1["objective_1"],
    "objective_2": best_obj1["objective_2"],
    "clashes": best_obj1["clashes"],
    "staff_days": best_obj1["staff_days"],
    "penalties": best_obj1["penalties"]
})

print("\nBest by objective 2:")
print({
    "objective_1": best_obj2["objective_1"],
    "objective_2": best_obj2["objective_2"],
    "clashes": best_obj2["clashes"],
    "staff_days": best_obj2["staff_days"],
    "penalties": best_obj2["penalties"]
})

Best by objective 1:
{'objective_1': 329, 'objective_2': 327, 'clashes': 29, 'staff_days': 27, 'penalties': 3}

Best by objective 2:
{'objective_1': 329, 'objective_2': 327, 'clashes': 29, 'staff_days': 27, 'penalties': 3}


In [33]:
def individual_summary(individual):
    return {
        "objective_1": individual["objective_1"],
        "objective_2": individual["objective_2"],
        "clashes": individual["clashes"],
        "staff_days": individual["staff_days"],
        "penalties": individual["penalties"]
    }

In [34]:
individual_summary(population[0])

{'objective_1': 929,
 'objective_2': 928,
 'clashes': 29,
 'staff_days': 28,
 'penalties': 9}

In [35]:
def dominates(individual_a, individual_b):
    """
    Return True if individual_a dominates individual_b.

    Minimization problem:
    - lower objective_1 is better
    - lower objective_2 is better
    """
    no_worse_in_all = (
        individual_a["objective_1"] <= individual_b["objective_1"] and
        individual_a["objective_2"] <= individual_b["objective_2"]
    )

    better_in_at_least_one = (
        individual_a["objective_1"] < individual_b["objective_1"] or
        individual_a["objective_2"] < individual_b["objective_2"]
    )

    return no_worse_in_all and better_in_at_least_one

In [36]:
print("Does population[0] dominate population[1]?",
      dominates(population[0], population[1]))

print("Does population[1] dominate population[0]?",
      dominates(population[1], population[0]))

Does population[0] dominate population[1]? True
Does population[1] dominate population[0]? False


In [37]:
def non_dominated_sort(population):
    """
    Perform non-dominated sorting on the population.

    Returns:
    - fronts: list of fronts, where each front is a list of individuals
    - each individual gets a 'rank' field
    """
    fronts = [[]]

    for p in population:
        p["dominated_solutions"] = []
        p["domination_count"] = 0

        for q in population:
            if p is q:
                continue

            if dominates(p, q):
                p["dominated_solutions"].append(q)
            elif dominates(q, p):
                p["domination_count"] += 1

        if p["domination_count"] == 0:
            p["rank"] = 0
            fronts[0].append(p)

    i = 0
    while len(fronts[i]) > 0:
        next_front = []

        for p in fronts[i]:
            for q in p["dominated_solutions"]:
                q["domination_count"] -= 1

                if q["domination_count"] == 0:
                    q["rank"] = i + 1
                    next_front.append(q)

        i += 1
        fronts.append(next_front)

    # Remove final empty front
    if len(fronts[-1]) == 0:
        fronts.pop()

    return fronts

In [38]:
fronts = non_dominated_sort(population)

print("Number of fronts:", len(fronts))
for i, front in enumerate(fronts[:5]):
    print(f"Front {i + 1} size:", len(front))

Number of fronts: 30
Front 1 size: 1
Front 2 size: 1
Front 3 size: 3
Front 4 size: 2
Front 5 size: 2


In [39]:
front_1_summaries = pd.DataFrame([
    {
        "objective_1": ind["objective_1"],
        "objective_2": ind["objective_2"],
        "clashes": ind["clashes"],
        "staff_days": ind["staff_days"],
        "penalties": ind["penalties"],
        "rank": ind["rank"]
    }
    for ind in fronts[0]
])

front_1_summaries

,objective_1,objective_2,clashes,staff_days,penalties,rank
0,329,327,29,27,3,0


In [40]:
front_sizes_df = pd.DataFrame({
    "front_number": list(range(1, len(fronts) + 1)),
    "size": [len(front) for front in fronts]
})

front_sizes_df

,front_number,size
0,1,1
1,2,1
2,3,3
3,4,2
4,5,2
5,6,1
6,7,2
7,8,1
8,9,2
9,10,2


In [41]:
ranked_population_df = pd.DataFrame([
    {
        "objective_1": ind["objective_1"],
        "objective_2": ind["objective_2"],
        "clashes": ind["clashes"],
        "staff_days": ind["staff_days"],
        "penalties": ind["penalties"],
        "rank": ind["rank"]
    }
    for ind in population
])

ranked_population_df.sort_values(by=["rank", "objective_1", "objective_2"]).head(15)

,objective_1,objective_2,clashes,staff_days,penalties,rank
36,329,327,29,27,3,0
17,531,525,31,25,5,1
6,532,530,32,30,5,2
31,538,528,38,28,5,2
46,539,526,39,26,5,2
47,627,627,27,27,6,3
35,629,625,29,25,6,3
23,632,627,32,27,6,4
34,633,626,33,26,6,4
9,635,626,35,26,6,5


In [42]:
def calculate_crowding_distance(front):
    """
    Assign crowding distance values to individuals in one Pareto front.

    Higher crowding distance means the solution is in a less crowded
    part of the objective space, so it should be preferred for diversity.
    """
    if len(front) == 0:
        return

    if len(front) == 1:
        front[0]["crowding_distance"] = float("inf")
        return

    if len(front) == 2:
        front[0]["crowding_distance"] = float("inf")
        front[1]["crowding_distance"] = float("inf")
        return

    # Initialize distances
    for individual in front:
        individual["crowding_distance"] = 0.0

    objectives = ["objective_1", "objective_2"]

    for objective in objectives:
        front.sort(key=lambda ind: ind[objective])

        # Boundary points always get infinite distance
        front[0]["crowding_distance"] = float("inf")
        front[-1]["crowding_distance"] = float("inf")

        min_value = front[0][objective]
        max_value = front[-1][objective]

        if max_value == min_value:
            continue

        for i in range(1, len(front) - 1):
            prev_value = front[i - 1][objective]
            next_value = front[i + 1][objective]

            distance = (next_value - prev_value) / (max_value - min_value)
            front[i]["crowding_distance"] += distance

In [43]:
def assign_crowding_distances(fronts):
    """
    Calculate crowding distance for every front.
    """
    for front in fronts:
        calculate_crowding_distance(front)

In [44]:
assign_crowding_distances(fronts)

In [45]:
front_1_crowding_df = pd.DataFrame([
    {
        "objective_1": ind["objective_1"],
        "objective_2": ind["objective_2"],
        "clashes": ind["clashes"],
        "staff_days": ind["staff_days"],
        "penalties": ind["penalties"],
        "rank": ind["rank"],
        "crowding_distance": ind["crowding_distance"]
    }
    for ind in fronts[0]
])

front_1_crowding_df

,objective_1,objective_2,clashes,staff_days,penalties,rank,crowding_distance
0,329,327,29,27,3,0,inf


In [46]:
def better_individual(individual_a, individual_b):
    """
    Return the better individual according to:
    1. lower Pareto rank
    2. higher crowding distance if ranks are equal
    """
    if individual_a["rank"] < individual_b["rank"]:
        return individual_a

    if individual_b["rank"] < individual_a["rank"]:
        return individual_b

    if individual_a["crowding_distance"] > individual_b["crowding_distance"]:
        return individual_a

    if individual_b["crowding_distance"] > individual_a["crowding_distance"]:
        return individual_b

    return random.choice([individual_a, individual_b])

In [47]:
def tournament_selection(population):
    """
    Perform binary tournament selection.
    Randomly choose two individuals and return the better one.
    """
    a, b = random.sample(population, 2)
    return better_individual(a, b)

In [48]:
selected_parent = tournament_selection(population)

{
    "objective_1": selected_parent["objective_1"],
    "objective_2": selected_parent["objective_2"],
    "clashes": selected_parent["clashes"],
    "staff_days": selected_parent["staff_days"],
    "penalties": selected_parent["penalties"],
    "rank": selected_parent["rank"],
    "crowding_distance": selected_parent["crowding_distance"]
}

{'objective_1': 629,
 'objective_2': 625,
 'clashes': 29,
 'staff_days': 25,
 'penalties': 6,
 'rank': 3,
 'crowding_distance': inf}

In [49]:
selected_parents = [tournament_selection(population) for _ in range(10)]

selected_parents_df = pd.DataFrame([
    {
        "objective_1": ind["objective_1"],
        "objective_2": ind["objective_2"],
        "clashes": ind["clashes"],
        "staff_days": ind["staff_days"],
        "penalties": ind["penalties"],
        "rank": ind["rank"],
        "crowding_distance": ind["crowding_distance"]
    }
    for ind in selected_parents
])

selected_parents_df

,objective_1,objective_2,clashes,staff_days,penalties,rank,crowding_distance
0,635,626,35,26,6,5,inf
1,834,828,34,28,8,10,inf
2,1030,1026,30,26,10,14,inf
3,635,626,35,26,6,5,inf
4,934,926,34,26,9,11,1.466667
5,832,824,32,24,8,8,inf
6,538,528,38,28,5,2,2.000000
7,1634,1626,34,26,16,27,inf
8,1030,1026,30,26,10,14,inf
9,539,526,39,26,5,2,inf
